In [1]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Project paths (the notebook lives in notebooks/)
ROOT = Path("..")
RAW_OSM = ROOT / "data" / "raw" / "osm"
INTERIM = ROOT / "data" / "interim"
RAW_OSM.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

with open(ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

COUNTRIES = config["scope"]["countries"]
PAUSE_S = config["osm"]["request_pause_s"]
print(len(COUNTRIES), "countries | pause:", PAUSE_S, "s")

30 countries | pause: 5 s


In [2]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def build_query(country_code: str) -> str:
    """Overpass QL query for data centres in one country (existing, under construction, planned)."""
    return f"""
    [out:json][timeout:180];
    area["ISO3166-1"="{country_code}"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    """

print(build_query("PT"))


    [out:json][timeout:180];
    area["ISO3166-1"="PT"][admin_level=2]->.country;
    (
      nwr["telecom"="data_center"](area.country);
      nwr["building"="data_center"](area.country);
      nwr["industrial"="data_centre"](area.country);
      nwr["construction:telecom"="data_center"](area.country);
      nwr["construction"="data_center"](area.country);
      nwr["proposed:telecom"="data_center"](area.country);
    );
    out geom;
    


In [3]:
HEADERS = {"User-Agent": "ia-agua-datacenters (portfolio project; github.com/Carol-Sust/Sustainability)"}

def fetch_country(country_code: str, force: bool = False, max_retries: int = 4) -> dict:
    """Download OSM data centres for one country, caching the raw JSON in data/raw/osm/."""
    cache_file = RAW_OSM / f"{country_code}.json"
    if cache_file.exists() and not force:
        return json.loads(cache_file.read_text())

    query = build_query(country_code)
    for attempt in range(1, max_retries + 1):
        response = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=300)
        if response.status_code == 200:
            data = response.json()
            cache_file.write_text(json.dumps(data))
            return data
        # 429 = too many requests, 504 = server busy -> wait and retry
        wait = 30 * attempt
        print(f"{country_code}: HTTP {response.status_code}, retry {attempt}/{max_retries} in {wait}s")
        time.sleep(wait)

    raise RuntimeError(f"{country_code}: failed after {max_retries} attempts")

In [4]:
pt = fetch_country("PT")
elements = pt["elements"]
print("Elements found:", len(elements))

# How many points (node), polygons (way) and relations?
print(pd.Series([e["type"] for e in elements]).value_counts())

# Tags of each element as a table
tags = pd.json_normalize([e.get("tags", {}) for e in elements])
cols = [c for c in ["name", "operator", "telecom", "building", "construction", "proposed:telecom"] if c in tags.columns]
tags[cols].head(20)

Elements found: 19
way     15
node     4
Name: count, dtype: int64


,name,operator,telecom,building
0,NaN,MEO,data_center,NaN
1,Centro de Computação Avançada do Minho,Fundação para a Ciência e Tecnologia,data_center,NaN
2,NaN,NaN,data_center,NaN
3,NaN,NaN,data_center,NaN
4,Equinix LS1,Equinix,data_center,data_center
5,NOS Imopolis II,NOS,data_center,data_center
6,NOS Imopolis I,NOS,data_center,data_center
7,Seixal Colocation Site,TATA Communications,data_center,yes
8,Data Center Portugal Telecom da Covilhã,Portugal Telecom,data_center,yes
9,Datacenter de Riba d'Ave,REN;NOS,data_center,data_center


In [5]:
from tqdm import tqdm

counts = {}
failed = []

for cc in tqdm(COUNTRIES):
    already_cached = (RAW_OSM / f"{cc}.json").exists()
    try:
        data = fetch_country(cc)
        counts[cc] = len(data["elements"])
    except RuntimeError as error:
        print(error)
        failed.append(cc)
    # Only pause after a real request, not when reading from cache
    if not already_cached:
        time.sleep(PAUSE_S)

print("Failed:", failed if failed else "none")

100%|████████████████████████████████████████| 30/30 [00:00<00:00, 1165.55it/s]

Failed: none


In [6]:
saved = sorted(p.stem for p in RAW_OSM.glob("*.json"))
missing = sorted(set(COUNTRIES) - set(saved))
print(len(saved), "files | missing:", missing if missing else "none")

30 files | missing: none


In [7]:
counts = {cc: len(json.loads((RAW_OSM / f"{cc}.json").read_text())["elements"]) for cc in saved}
summary = pd.Series(counts, name="elements").sort_values(ascending=False)
print("Total elements:", summary.sum())
summary

Total elements: 1809


FR    390
DE    320
GB    310
NL    170
IE    113
IT     75
ES     52
BE     41
FI     38
SE     35
CH     35
PL     34
DK     26
CZ     24
HU     23
NO     21
PT     19
AT     17
HR     13
LU     10
GR     10
LV      6
RO      6
BG      6
LT      5
SK      5
SI      3
EE      2
MT      0
CY      0
Name: elements, dtype: int64

In [8]:
saved = sorted(p.stem for p in RAW_OSM.glob("*.json"))
counts = {cc: len(json.loads((RAW_OSM / f"{cc}.json").read_text())["elements"]) for cc in saved}
summary = pd.Series(counts, name="elements").sort_values(ascending=False)
print(len(saved), "countries | total elements:", summary.sum())
summary

30 countries | total elements: 1809


FR    390
DE    320
GB    310
NL    170
IE    113
IT     75
ES     52
BE     41
FI     38
SE     35
CH     35
PL     34
DK     26
CZ     24
HU     23
NO     21
PT     19
AT     17
HR     13
LU     10
GR     10
LV      6
RO      6
BG      6
LT      5
SK      5
SI      3
EE      2
MT      0
CY      0
Name: elements, dtype: int64

In [9]:
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union

CRS_GEO = config["crs"]["geographic"]      # EPSG:4326
CRS_AREA = config["crs"]["equal_area"]     # EPSG:3035

In [10]:
def way_to_geometry(coords):
    """A list of (lon, lat) points -> Polygon if closed, otherwise LineString."""
    if len(coords) >= 4 and coords[0] == coords[-1]:
        return Polygon(coords)
    if len(coords) >= 2:
        return LineString(coords)
    return Point(coords[0])

def element_geometry(element):
    """Build a shapely geometry from an Overpass element (out geom)."""
    if element["type"] == "node":
        return Point(element["lon"], element["lat"])

    if element["type"] == "way":
        coords = [(p["lon"], p["lat"]) for p in element.get("geometry", [])]
        return way_to_geometry(coords) if coords else None

    if element["type"] == "relation":
        # Combine the outer members of the relation (e.g. a campus with several buildings)
        parts = []
        for member in element.get("members", []):
            if member.get("role") in ("outer", "") and "geometry" in member:
                coords = [(p["lon"], p["lat"]) for p in member["geometry"]]
                if coords:
                    parts.append(way_to_geometry(coords))
        return unary_union(parts) if parts else None

    return None

In [11]:
def element_status(tags):
    """existing / under_construction / planned, based on OSM lifecycle tags."""
    if tags.get("proposed:telecom") == "data_center" or tags.get("landuse") == "proposed":
        return "planned"
    if (
        tags.get("construction:telecom") == "data_center"
        or tags.get("construction") == "data_center"
        or tags.get("landuse") == "construction"
        or tags.get("building") == "construction"
    ):
        return "under_construction"
    return "existing"

In [12]:
rows = []
for cc in COUNTRIES:
    data = json.loads((RAW_OSM / f"{cc}.json").read_text())
    for element in data["elements"]:
        tags = element.get("tags", {})
        rows.append({
            "osm_type": element["type"],
            "osm_id": element["id"],
            "country": cc,
            "name": tags.get("name"),
            "operator": tags.get("operator"),
            "status": element_status(tags),
            "geometry": element_geometry(element),
        })

gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_GEO)

# Remove elements without geometry and any duplicates
gdf = gdf.dropna(subset=["geometry"]).drop_duplicates(subset=["osm_type", "osm_id"])

# Area in m² (only for polygons; points and lines get NaN)
is_polygon = gdf.geom_type.isin(["Polygon", "MultiPolygon"])
gdf["area_m2"] = gdf.to_crs(CRS_AREA).area.where(is_polygon)

# Centroid as a point, computed in metres for accuracy and converted back to lat/lon
centroids = gdf.to_crs(CRS_AREA).centroid.to_crs(CRS_GEO)
gdf["lon"] = centroids.x
gdf["lat"] = centroids.y

print(len(gdf), "elements")
gdf.head()

1808 elements


,osm_type,osm_id,country,name,operator,status,geometry,area_m2,lon,lat
0,node,6578896177,AT,earthDATAsafe,Kapsch BusinessCom AG,existing,POINT (15.33817 47.44914),NaN,15.338173,47.449138
1,node,12596690617,AT,IT- und Data Center Linz,LINZ NETZ GmbH,existing,POINT (14.30832 48.28215),NaN,14.308318,48.282151
2,node,13092390748,AT,IT- und Data Center Linz,NaN,existing,POINT (14.3 48.3242),NaN,14.299996,48.324198
3,way,31990236,AT,Digital Realty VIE1 & VIE2,Digital Realty,existing,"POLYGON ((16.40878 48.26918, 16.40867 48.26874...",27721.168009,16.410523,48.268928
4,way,74662577,AT,Telekom-Tower,A1 / Telekom Austria,existing,"POLYGON ((15.43 47.06632, 15.42999 47.06619, 1...",870.136764,15.430370,47.066245


In [13]:
print(gdf["status"].value_counts(), "\n")
print(gdf.geom_type.value_counts(), "\n")
print(gdf["area_m2"].describe().round(0))

status
existing              1759
under_construction      38
planned                 11
Name: count, dtype: int64 

Polygon               1421
Point                  372
MultiLineString          9
MultiPolygon             5
GeometryCollection       1
Name: count, dtype: int64 

count      1426.0
mean      12128.0
std       47334.0
min           3.0
25%        1300.0
50%        3557.0
75%        8227.0
max      875069.0
Name: area_m2, dtype: float64


In [14]:
from shapely.ops import polygonize

def lines_to_polygon(geom):
    """Close line segments into polygons (for relations whose outline came in pieces)."""
    if geom.geom_type in ("Polygon", "MultiPolygon", "Point"):
        return geom
    parts = list(getattr(geom, "geoms", [geom]))
    lines = [p for p in parts if p.geom_type in ("LineString", "LinearRing", "MultiLineString")]
    polygons = [p for p in parts if p.geom_type in ("Polygon", "MultiPolygon")]
    polygons += list(polygonize(lines))
    return unary_union(polygons) if polygons else geom.centroid

gdf["geometry"] = gdf.geometry.apply(lines_to_polygon)

# Recompute area and centroid with the fixed geometries
is_polygon = gdf.geom_type.isin(["Polygon", "MultiPolygon"])
gdf["area_m2"] = gdf.to_crs(CRS_AREA).area.where(is_polygon)
centroids = gdf.to_crs(CRS_AREA).centroid.to_crs(CRS_GEO)
gdf["lon"], gdf["lat"] = centroids.x, centroids.y

print(gdf.geom_type.value_counts())

Polygon         1430
Point            372
MultiPolygon       6
Name: count, dtype: int64


In [15]:
cols = ["country", "name", "operator", "status", "osm_type", "area_m2"]
print("Largest:")
display(gdf.nlargest(10, "area_m2")[cols])
print("Smallest:")
display(gdf.nsmallest(10, "area_m2")[cols])

Largest:


,country,name,operator,status,osm_type,area_m2
166,FI,Alussuo,Tuike Finland Oy,under_construction,relation,3.312547e+06
1021,IE,Meta Clonee Data Center,NaN,existing,relation,1.368472e+06
142,FI,Google Haminan palvelinkeskus,Google,existing,way,8.750690e+05
32,BE,Google Data Center,Google,existing,way,8.467917e+05
1282,NL,Microsoft Datacenter,Microsoft,existing,way,4.996921e+05
1022,IE,Microsoft Grange Castle,Microsoft,existing,relation,4.976187e+05
1442,SE,EcoDataCenter,NaN,under_construction,relation,4.878489e+05
111,DK,Google Data Center Fredericia,Google,existing,way,4.788082e+05
1253,NL,Google Datacenter Eemshaven,Google,existing,way,4.341921e+05
1245,NL,Microsoft Datacenter West Europe,Microsoft,existing,way,3.530679e+05


Smallest:


,country,name,operator,status,osm_type,area_m2
1257,NL,NaN,NaN,existing,way,2.597401
1767,NO,NaN,NaN,existing,way,4.479545
1758,NO,Signal,NaN,existing,way,6.654209
1768,NO,NaN,NaN,existing,way,7.086897
1385,ES,NaN,NaN,existing,way,10.184668
1384,ES,centro de datos trasbase Negratín,NaN,existing,way,10.998098
1337,PT,NaN,NaN,existing,way,11.496762
660,DE,Dernbach 1,GasLINE,existing,way,12.238588
1354,SI,NaN,NaN,existing,way,12.676986
1070,IT,NaN,NaN,existing,way,12.925483


In [16]:
for threshold in [25, 50, 100, 200, 500]:
    n = (gdf["area_m2"] < threshold).sum()
    print(f"< {threshold:>3} m²: {n:>4} elements ({n / gdf['area_m2'].notna().sum():.1%} of polygons)")

<  25 m²:   21 elements (1.5% of polygons)
<  50 m²:   28 elements (1.9% of polygons)
< 100 m²:   52 elements (3.6% of polygons)
< 200 m²:   65 elements (4.5% of polygons)
< 500 m²:  139 elements (9.7% of polygons)


In [17]:
MIN_AREA_M2 = config["osm"]["min_area_m2"]

before = len(gdf)
too_small = gdf["area_m2"] < MIN_AREA_M2      # points have NaN area -> False -> kept
gdf = gdf[~too_small].copy()

print(f"Min area: {MIN_AREA_M2} m²")
print(f"Removed: {before - len(gdf)} elements | Remaining: {len(gdf)}")
print("Smallest area now:", round(gdf["area_m2"].min(), 1), "m²")

Min area: 100 m²
Removed: 52 elements | Remaining: 1756
Smallest area now: 106.1 m²


In [18]:
CAMPUS_DIST = config["osm"]["campus_distance_m"]

g = gdf.to_crs(CRS_AREA).copy()   # work in metres
parts = []

for status, sub in g.groupby("status"):
    # Buffer each element by half the distance: two elements < 200 m apart -> their buffers touch
    blobs = gpd.GeoDataFrame(
        geometry=[unary_union(sub.geometry.buffer(CAMPUS_DIST / 2))], crs=CRS_AREA
    ).explode(index_parts=False).reset_index(drop=True)
    blobs["campus_id"] = [f"{status}_{i}" for i in range(len(blobs))]

    # Each element gets the id of the blob it falls in
    joined = gpd.sjoin(sub, blobs, how="left", predicate="intersects")
    parts.append(joined.drop(columns="index_right"))

g = pd.concat(parts)
print(len(g), "elements ->", g["campus_id"].nunique(), "campuses")

1756 elements -> 1124 campuses


In [19]:
def most_common(values):
    """Most frequent non-empty value (e.g. the campus name), or None."""
    values = values.dropna()
    return values.mode().iloc[0] if len(values) else None

campuses = g.groupby("campus_id").agg(
    country=("country", most_common),
    name=("name", most_common),
    operator=("operator", most_common),
    operators=("operator", lambda s: "; ".join(sorted(s.dropna().unique()))),
    n_operators=("operator", lambda s: s.dropna().nunique()),
    status=("status", "first"),
    n_elements=("osm_id", "size"),
)

# Merge the shapes of each campus into one geometry
shapes = g.dissolve(by="campus_id")[["geometry"]]
campuses = gpd.GeoDataFrame(campuses.join(shapes), geometry="geometry", crs=CRS_AREA)

# Area of the merged shape (overlapping polygons are not counted twice); points-only campuses -> NaN
campuses["area_m2"] = campuses.area.where(campuses.area > 0)

# Centroid in lat/lon
centroids = campuses.centroid.to_crs(CRS_GEO)
campuses["lon"], campuses["lat"] = centroids.x, centroids.y

campuses = campuses.to_crs(CRS_GEO).reset_index()
print(len(campuses), "campuses")
campuses.head()

1124 campuses


,campus_id,country,name,operator,operators,n_operators,status,n_elements,geometry,area_m2,lon,lat
0,existing_0,ES,Aire Canarias,AIRE,AIRE,1,existing,1,"POLYGON ((-15.42928 28.11871, -15.42925 28.118...",265.460434,-15.429130,28.118717
1,existing_1,ES,inforland.com,inforland.com,inforland.com,1,existing,1,POINT (-3.87738 36.74839),NaN,-3.877379,36.748388
2,existing_10,PT,Data Center IPT,NaN,,0,existing,1,"POLYGON ((-8.38868 39.59981, -8.38853 39.59977...",193.460168,-8.388629,39.599728
3,existing_100,GB,NaN,Ionos,Ionos,1,existing,1,"POLYGON ((-2.1575 52.20592, -2.15629 52.20568,...",4067.789146,-2.156672,52.205932
4,existing_1000,GR,COSMOTE,NaN,,0,existing,1,"POLYGON ((19.91892 39.6198, 19.91899 39.61966,...",453.470771,19.919106,39.619776


In [20]:
print(campuses["status"].value_counts(), "\n")
print("Elements per campus:")
print(campuses["n_elements"].value_counts().sort_index().head(10), "\n")

cols = ["country", "name", "operator", "status", "n_elements", "area_m2"]
campuses.nlargest(10, "n_elements")[cols]

status
existing              1082
under_construction      31
planned                 11
Name: count, dtype: int64 

Elements per campus:
n_elements
1     892
2     125
3      39
4      26
5       9
6       9
7       5
8       3
9       4
10      2
Name: count, dtype: int64 



,country,name,operator,status,n_elements,area_m2
405,IE,EdgeConneX,Microsoft,existing,29,694037.210583
839,DE,Digital Realty FRA1,Digital Realty,existing,24,138323.645357
151,GB,Equinix LD10,Equinix,existing,21,148409.973920
315,FR,CNRS,data4,existing,20,349825.409382
597,NL,Digital Realty AMS11,EdgeConneX,existing,19,84700.001829
472,IE,Amazon Web Services,Amazon Web Services,existing,14,214805.808754
787,DE,NTT,NTT,existing,14,108757.504610
922,DE,Hetzner – Rechenzentrumspark Falkenstein,Hetzner Online GmbH,existing,14,175738.579412
781,DE,Equinix FR5,Telehouse,existing,13,25359.178140
786,DE,Colt FRA3,Digital Realty,existing,11,38731.790400


In [21]:
print((campuses["n_operators"] > 1).sum(), "sites with more than one operator")
campuses[campuses["n_operators"] > 1][["country", "name", "operators", "n_elements"]].head(10)

73 sites with more than one operator


,country,name,operators,n_elements
31,HU,BIX,ATW; ISZT - BUDAPEST INTERNET EXCHANGE,2
54,PL,Data4 Pologne WAW1,Data4; Netia,6
93,FI,CSC – IT Center for Science,CSC; EuroHPC/CSC,2
94,IE,Amazon Web Services,Amazon Web Services; Digital Realty; EXA Infra...,9
97,IE,Amazon Web Services,Amazon Web Services; Verizon; Vodafone,10
111,GB,Pulsant Edinburgh South Gyle SC-1,Lloyds Banking Group; Pulsant,3
117,ES,MAD1,Digital Realty; iaas,3
126,GB,Astrohouse,SCC; Serverhouse; University of Southampton,3
131,GB,Datum Farnborough Data Centre,Ark Data Centres; Datum,7
134,GB,Gravity Edge LDN1,Bank Of America; Gravity Edge,2


In [22]:
by_country = (
    campuses.pivot_table(index="country", columns="status", values="campus_id",
                         aggfunc="count", fill_value=0)
    .assign(total=lambda t: t.sum(axis=1))
    .sort_values("total", ascending=False)
)
by_country

status,existing,planned,under_construction,total
country,,,,
FR,263,1,6,270
GB,191,8,7,206
DE,164,1,9,174
NL,75,0,2,77
IT,51,0,0,51
ES,34,0,1,35
BE,32,0,0,32
IE,27,1,2,30
CH,27,0,0,27


In [23]:
sample = campuses.sample(20, random_state=42)[["country", "name", "operators", "status", "lat", "lon"]].copy()
sample["map"] = sample.apply(
    lambda r: f"https://www.openstreetmap.org/?mlat={r.lat:.5f}&mlon={r.lon:.5f}#map=17/{r.lat:.5f}/{r.lon:.5f}", axis=1
)
sample

,country,name,operators,status,lat,lon,map
1088,IE,NaN,,planned,53.410841,-6.320308,https://www.openstreetmap.org/?mlat=53.41084&m...
896,IT,Centrale TIM Trento Nord,TIM,existing,46.084651,11.115079,https://www.openstreetmap.org/?mlat=46.08465&m...
739,CH,24 hours fiberwork ag,,existing,47.064375,8.277522,https://www.openstreetmap.org/?mlat=47.06438&m...
140,GB,Keppel Data Centres (Vodafone) - London,Vodafone,existing,51.416527,-0.772348,https://www.openstreetmap.org/?mlat=51.41653&m...
1041,DE,NaN,NTT Global Data Centers EMEA GmbH,existing,52.438451,13.374489,https://www.openstreetmap.org/?mlat=52.43845&m...
527,PT,Seixal Colocation Site,TATA Communications,existing,38.600025,-9.159862,https://www.openstreetmap.org/?mlat=38.60002&m...
812,IT,Equinix Milan ML3,Equinix,existing,45.354055,9.147331,https://www.openstreetmap.org/?mlat=45.35406&m...
481,FR,Datacenter Sud,Aix-Marseille Université,existing,43.335920,5.411936,https://www.openstreetmap.org/?mlat=43.33592&m...
1117,FI,Alussuo,Tuike Finland Oy,under_construction,64.107351,27.157622,https://www.openstreetmap.org/?mlat=64.10735&m...
949,FR,Data Center Bluecom,Bluecom,existing,48.441301,-4.415470,https://www.openstreetmap.org/?mlat=48.44130&m...


In [24]:
# Sites (one row per data centre site) -> used in the next steps
campuses.to_parquet(INTERIM / "osm_datacenters.parquet")

# Individual OSM elements (buildings) -> kept for reference
gdf.to_parquet(INTERIM / "osm_elements.parquet")

# Read back to check
check = gpd.read_parquet(INTERIM / "osm_datacenters.parquet")
print(len(check), "sites saved |", check.crs)
check.head(3)

1124 sites saved | {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "

,campus_id,country,name,operator,operators,n_operators,status,n_elements,geometry,area_m2,lon,lat
0,existing_0,ES,Aire Canarias,AIRE,AIRE,1,existing,1,"POLYGON ((-15.42928 28.11871, -15.42925 28.118...",265.460434,-15.429130,28.118717
1,existing_1,ES,inforland.com,inforland.com,inforland.com,1,existing,1,POINT (-3.87738 36.74839),NaN,-3.877379,36.748388
2,existing_10,PT,Data Center IPT,NaN,,0,existing,1,"POLYGON ((-8.38868 39.59981, -8.38853 39.59977...",193.460168,-8.388629,39.599728
